# 降水量予報を補正する U-Net — 用語と手法の解説

このノートブックは、`precip_unet_lab` で使っている**機械学習の手法と専門用語**を、手を動かしながら理解するための解説です。
上から順に実行してください（全体で数分、CPU のみ・合成データのみで動きます）。

## 取り組んでいる問題
> 粗い格子（約 80 km 四方）の**降水量予報**を、細かい格子（約 10 km 四方）の降水量に直したい。

これは次の 2 つを同時に行う作業です。
- **ダウンスケーリング**: 粗い格子の情報から、細かい格子の分布を推定する。
- **バイアス補正**: 予報の系統的なずれ（雨を多く／少なく出す癖）を直す。

## 手法の全体像
```
粗い予報 ──(保存的再格子化)──▶ ベース場（細かい格子に載せただけ）
ベース場 ──(U-Net が r を出す)──▶ 予測 = ベース場 × exp(r)     ← 乗法的補正（対数空間の残差学習）
予測 と 真の降水 ──▶ 損失関数で差を測り、U-Net の重みを更新
評価 ──▶ 指標を複数使い、ブートストラップで「差が誤差でないか」を確かめる
```

## 目次
1. 降水データの特徴
2. 再格子化（保存的 と 双線形）
3. U-Net の部品（畳み込み・GroupNorm・SiLU・プーリング・拡大・スキップ結合）と、U-Net の考え方・CNN との関係
4. 乗法的補正と、対数空間の残差学習
5. 損失関数と「総量が落ちる」問題
6. 評価指標（RMSE・対数 RMSE・総量比・POD・FAR・CSI・BI・FSS）と、CSI の読み方
7. ブートストラップ（信頼区間）
8. 実データ（GSM × GSMaP）での注意点と、時間順のブロック分割・QM と期間のずれ
9. 用語集

## 0. 準備

In [ ]:
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")          # このリポジトリの src/ を読み込む
plt.rcParams["font.family"] = ["Hiragino Sans", "Arial Unicode MS", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)
np.random.seed(0)
print("torch", torch.__version__)

## 1. 降水データの特徴

降水量（単位: **mm/h** = 1 時間あたりの雨量）のデータには、普通の画像と違う癖があります。

| 特徴 | 内容 | 学習で困る点 |
|---|---|---|
| **ゼロが多い** | ほとんどの場所・時刻で雨は降らない | 「全部 0」と答えるだけでそこそこ当たってしまう |
| **歪んだ分布** | 弱い雨が大半で、強い雨は稀だが被害が大きい | 強い雨が少数で、学習で軽く扱われやすい |
| **総量が大事** | 領域の降水の合計（総量）は水収支に直結 | 見た目が良くても総量が狂うと使えない |

ここでは本物のデータではなく、**合成データ**（`src/synth.py`）で練習します。
真の降水場を自分で作るので、**手法が正解に近づいたかを正確に測れる**のが利点です。
- `fine`: 真の細かい降水場（64×64 格子）
- `coarse`: 誤差入りの粗い予報（8×8 格子 = 8 倍粗い。過大バイアス・位置ずれ・ばらつきを含む）

In [ ]:
from synth import make_dataset

coarse, fine = make_dataset(200, n=64, factor=8, seed=1)
idx = [i for i in range(len(fine)) if fine[i].max() > 5][:4]     # 雨のある事例を4つ

fig, ax = plt.subplots(2, 4, figsize=(14, 6.5))
for j, i in enumerate(idx):
    ax[0, j].imshow(fine[i], vmin=0, vmax=15, cmap="Blues")
    ax[0, j].set_title(f"真の降水（64×64）事例{i}")
    ax[1, j].imshow(coarse[i], vmin=0, vmax=15, cmap="Blues")
    ax[1, j].set_title("粗い予報（8×8）")
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

v = fine.ravel()
print(f"雨が全くない画素の割合: {(v == 0).mean():.1%}")
print(f"1 mm/h 以上: {(v >= 1).mean():.1%}   5 mm/h 以上: {(v >= 5).mean():.2%}   10 mm/h 以上: {(v >= 10).mean():.3%}")
plt.figure(figsize=(6, 3))
plt.hist(v[v > 0], bins=60, log=True)
plt.xlabel("降水強度 [mm/h]（雨のある画素のみ）"); plt.ylabel("画素数（対数軸）")
plt.title("弱い雨が大半で、強い雨は急速に減る（歪んだ分布）"); plt.show()

## 2. 再格子化（粗い格子 ⇄ 細かい格子）

格子の細かさが違うデータを重ねるには、**再格子化**（格子の貼り替え）が要ります。降水ではやり方で結果が変わります。

- **保存的再格子化**: 出力セルの値を、入力セルとの**重なり面積で重み付けした平均**にする。
  → 領域全体の平均（＝**総量**）が必ず保たれる。降水の基準にする。
- **双線形補間**: 周囲 4 点から直線的に補う。なめらかな連続量（気温・地形）向き。
  → 尖った場を**粗くする向き**で使うと、総量が変わってしまう。降水には使わない。

In [ ]:
from regrid import regrid_conservative, regrid_bilinear, conservative_matrix

# 総量の保存: 真の場(64×64)を 8×8 に粗くする
a = fine[idx[0]]
print("元の面積平均        :", round(a.mean(), 4))
print("保存的で粗くした平均:", round(regrid_conservative(a, (8, 8)).mean(), 4), " ← 一致")
print("双線形で粗くした平均:", round(regrid_bilinear(a, (8, 8)).mean(), 4), " ← 元と違う")

# 尖った場（1点だけの強い雨）で差が一番はっきり出る
p = np.zeros((6, 6)); p[2, 2] = 10.0
print("\n1点だけ雨のある場 → 4×4 に粗く")
print("  元の平均 :", round(p.mean(), 4))
print("  保存的   :", round(regrid_conservative(p, (4, 4)).mean(), 4))
print("  双線形   :", round(regrid_bilinear(p, (4, 4)).mean(), 4), "（雨を取りこぼす）")

# 保存的再格子化の「重み」を見る（1次元・3セル→2セル）
print("\n3セル→2セルの重み行列（各行の和は1）")
print(np.round(conservative_matrix(3, 2), 3))

### 2-1. なぜ降水に双線形を使わないのか（くわしく）
**粗くする向き**で、細かい 8 マスのうち 1 マスだけ 8 mm/h、他は 0 の場を、2 マスに粗くします（元の平均は 1.0）。
- 保存的: 新しいマスと重なる元のマスを**すべて**面積で平均する → 雨のかたまりがどこにあっても平均は同じ。
- 双線形: 新しいマスの**中心の近くの点だけ**を読む → かたまりが当たれば何倍にも増え、外れれば消える。

In [ ]:
from regrid import conservative_matrix as Cm, bilinear_matrix as Bm

print("細かい8マス → 粗い2マス（元の平均は 1.0）")
for pos in (1, 3):
    f = np.zeros(8); f[pos] = 8.0
    c, b = Cm(8, 2) @ f, Bm(8, 2) @ f
    print(f"  雨が左から{pos + 1}番目: 保存的 {c}（平均 {c.mean():.1f}）   双線形 {b}（平均 {b.mean():.1f}）")
print("→ 保存的は位置によらず平均 1.0。双線形は 2 倍に増えたり 0 に消えたりする")

**細かくする向き**（割り切れる比）では、双線形でも総量がたまたま保たれることがあります。ただし山の頂上が削られ、強い雨が薄まります。
だから降水は、向きによらず**保存的**に統一します。双線形は、気温・標高のような連続量と、U-Net のデコーダで**特徴マップ**を拡大する所だけで使います。

**欠測（データがない所）との関係**: 双線形も保存的も、欠測は扱えません。欠測は別に**マスク**（この画素は損失・評価から外す印）で除きます。
書籍などに出てくる「欠測を考慮した双線形補間」は別の派生版で、**有効な点だけで重みを再正規化**します（例: 重み 0.4・0.3・0.2・0.1 のうち 0.1 が欠測なら、残りの和 0.9 で割り直す）。
ただし、これも合計は保存しないので、降水には使いません。欠測を 0 mm/h とみなすと「雨が降らなかった」と誤って学ぶので、**マスクで除く**ことが大切です。

In [ ]:
v = np.array([0.4, 0.3, 0.2, 0.1])             # 周囲4点の重み
val = np.array([2.0, 4.0, 6.0, np.nan])        # 4点目が欠測
naive = np.nansum(v * val)                      # 欠測を 0 とみなす（値が小さくなる）
valid = ~np.isnan(val)
renorm = (v[valid] * val[valid]).sum() / v[valid].sum()   # 有効な点だけで重みを再正規化
print(f"欠測を0とみなす: {naive:.2f}   有効な点で再正規化: {renorm:.2f}")

今回の手法の出発点は、粗い予報を細かい格子に**保存的に載せた「ベース場」**です。ベース場は総量が予報と同じで、ブロック状（階段状）に見えます。
U-Net の仕事は、このベース場を「真の降水」に近づける**補正**を出すことです。

In [ ]:
base = regrid_conservative(coarse[idx[0]], (64, 64))
print("粗い予報の平均:", round(coarse[idx[0]].mean(), 4), " ベース場の平均:", round(base.mean(), 4))
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a_, d, t in zip(ax, [coarse[idx[0]], base, fine[idx[0]]], ["粗い予報 8×8", "ベース場 64×64（保存的）", "真の降水 64×64"]):
    a_.imshow(d, vmin=0, vmax=15, cmap="Blues"); a_.set_title(t); a_.set_xticks([]); a_.set_yticks([])
plt.show()

## 3. U-Net の部品

**U-Net** は、画像を「縮めて全体を見る → 元の大きさに戻す」を 1 つのネットワークで行う構造です（形が U 字なのでこの名前）。
画像から画像を作る問題（セグメンテーション・補正）でよく使われます。部品を 1 つずつ見ます。

```
入力 ─ConvBlock─┬──────────────(スキップ結合)──────────┐
                ▼ 平均プーリング                          ▼ 連結
              ConvBlock─┬───────(スキップ結合)────┐    ConvBlock ─▶ 出力 r
                        ▼ 平均プーリング           ▼ 連結
                      ConvBlock ─ 双線形拡大 ─────┘
```

### 3-1. 3×3 畳み込み（convolution）
小さな窓（3×3）を画像の上でずらしながら、**重みとの積和**を取る操作です。周囲の画素との関係（縁・広がり）を捉えます。
重みが学習される部分で、ここでは「縦の縁を見つける」重みを手で与えて動きを見ます。

In [ ]:
x = torch.zeros(1, 1, 6, 6); x[0, 0, :, 3:] = 1.0          # 右半分が 1 の画像（縦の縁がある）
k = torch.tensor([[-1., 0., 1.]] * 3).reshape(1, 1, 3, 3)   # 左右の差を取る重み
y = F.conv2d(x, k, padding=1)                                # padding=1: 縁の外を0で埋め、大きさを保つ
print("入力"); print(x[0, 0].int().numpy())
print("畳み込み後（縁の所だけ大きい値）"); print(y[0, 0].int().numpy())
print("\nnn.Conv2d は、この重みを学習で自動的に決める。 例: パラメータ数 =", sum(p.numel() for p in nn.Conv2d(1, 8, 3).parameters()), "(= 8×(3×3)+8)")

### 3-2. グループ正規化（GroupNorm）
チャンネル（特徴の種類）をいくつかのグループに分け、**グループごとに平均 0・分散 1 にそろえる**操作です。
値の大きさが層ごとにばらつくと学習が不安定になるので、それをならします。
バッチ正規化と違い、**バッチの大きさに依存しない**ため、少ない事例・小さなバッチ（本プロジェクトは CPU で小バッチ）でも安定します。

In [ ]:
x = torch.randn(4, 8, 16, 16) * 5 + 10                      # 平均10・標準偏差5 の特徴（4事例・8チャンネル）
gn = nn.GroupNorm(num_groups=2, num_channels=8)               # 8チャンネルを 2グループ(4ch ずつ)に
with torch.no_grad():
    y = gn(x)
print("正規化前: 平均 %.2f  標準偏差 %.2f" % (x.mean(), x.std()))
print("正規化後: 平均 %.2f  標準偏差 %.2f" % (y.mean(), y.std()))

### 3-3. SiLU（活性化関数）
畳み込みは線形（足し算と掛け算だけ）なので、そのままでは何層重ねても 1 層と同じです。
そこで間に**非線形な関数**を挟みます。SiLU は `x · sigmoid(x)` で、ReLU（負を 0 にする）より**なめらか**で、負の値も少し通します。

In [ ]:
xs = torch.linspace(-5, 5, 200)
plt.figure(figsize=(5, 3))
plt.plot(xs, F.relu(xs), label="ReLU", ls="--"); plt.plot(xs, F.silu(xs), label="SiLU = x·sigmoid(x)")
plt.axhline(0, c="gray", lw=0.5); plt.legend(); plt.title("活性化関数"); plt.show()

### 3-4. 平均プーリング と 3-5. 双線形拡大
- **平均プーリング**: 2×2 などの範囲を**平均して縮める**。画像が半分の大きさになり、より広い範囲を見られる（広い視野）。
  降水では面積平均なので、**総量を保つ**縮め方でもある。
- **双線形拡大**: 縮めた特徴を、双線形補間で元の大きさに戻す。デコーダ（復元側）で使う。これは**特徴マップ**の拡大であり、降水そのものの再格子化ではない（降水には保存的再格子化を使う）。

In [ ]:
a = torch.arange(16.).reshape(1, 1, 4, 4)
p = F.avg_pool2d(a, 2)
u = F.interpolate(p, scale_factor=2, mode="bilinear", align_corners=False)
print("元(4×4)\n", a[0, 0].numpy()); print("平均プーリング(2×2)\n", p[0, 0].numpy()); print("双線形拡大(4×4)\n", np.round(u[0, 0].numpy(), 2))

### 3-6. スキップ結合
縮めて戻す間に、**細かい情報（位置・縁）が失われる**ので、縮める前の特徴を、戻した特徴に**チャンネル方向に連結（concat）**して渡します。
「全体の文脈（縮めた側）」と「細部（縮める前）」の両方を最後の層で使えます。

In [ ]:
from model import ConvBlock, UNet

deep = torch.rand(1, 16, 8, 8)                                   # 縮めて処理した特徴（粗い）
shallow = torch.rand(1, 8, 16, 16)                               # 縮める前の特徴（細かい）
up = F.interpolate(deep, scale_factor=2, mode="bilinear", align_corners=False)
cat = torch.cat([up, shallow], dim=1)                            # スキップ結合 = チャンネルで連結
print("拡大後", tuple(up.shape), "+ スキップ", tuple(shallow.shape), "→ 連結", tuple(cat.shape))

print("\nConvBlock = (3×3畳み込み → GroupNorm → SiLU) × 2")
print(ConvBlock(1, 8))
net = UNet(c_in=1, base=8, depth=3, c_out=1)
y = net(torch.rand(2, 1, 64, 64))
print("\nU-Net の入出力の形:", tuple(y.shape), " パラメータ数:", sum(p.numel() for p in net.parameters()))

### 3-7. U-Net をやさしく言うと
地図を見るときを思い浮かべてください。
- **広い範囲を見る**（地図を引いて眺める）: 「あの雨雲の塊が北へ動いている」といった**大きな構造**が分かる。でも、細かい場所は分からない。
- **細かい所を見る**（地図を拡大する）: 「この 1 マスで降っている」が分かる。でも、全体の様子は分からない。

降水の補正には**両方**が要ります。U-Net は、**縮めて広く見る → 戻して細かく見る**を 1 つのネットワークで行い、
縮める前の細かい情報を**スキップ結合**で渡して、最後に両方を組み合わせます。名前は、構造を図にすると U の字になることに由来します。

### 3-8. CNN との関係
**CNN（畳み込みニューラルネットワーク）**は、畳み込み層を重ねたネットワークの総称です。U-Net は **CNN の一種**（畳み込み層の積み重ね＋縮小・拡大・スキップ結合）です。

| | 普通の CNN（画像分類など） | 全畳み込みの CNN（畳み込みだけを重ねる） | U-Net |
|---|---|---|---|
| 出力 | 画像全体に 1 つのラベル（例: 「猫」） | 入力と同じ大きさの画像 | 入力と同じ大きさの画像 |
| 構造 | 畳み込み＋縮小を重ね、最後に 1 つの値へ | 縮小せず畳み込みだけ | **縮小して戻し、スキップ結合を持つ** |
| 見える範囲（受容野） | 縮めるほど広い | 層の数に比例して狭い | **縮小で広くなり、細部も保てる** |
| 向く問題 | 分類 | 小さな範囲で決まる処理 | **画像から画像へ**（補正・領域分割） |

「**受容野**」とは、出力の 1 画素が、入力のどの範囲を見て決まっているかです。下で、同じくらいの数の 3×3 畳み込みを持つネットワークの受容野を、理論値で比べます（目安）。

In [ ]:
def receptive_field(layers):
    # conv: 3×3 畳み込み（見える範囲が広がる）  pool: 2×2 プーリング（以後の1歩が2倍の幅）  up: 2倍拡大（1歩の幅が半分）
    rf, jump = 1, 1
    for kind in layers:
        if kind == "conv":
            rf += 2 * jump
        elif kind == "pool":
            rf += jump; jump *= 2
        elif kind == "up":
            jump //= 2
    return rf

plain = ["conv"] * 14                                                          # 縮小なしで 14 層
unet = (["conv"] * 2 + ["pool"]) * 3 + ["conv"] * 2 + ["up"] + (["conv"] * 2 + ["up"]) * 2 + ["conv"] * 2   # U-Net(深さ3)の最も深い経路
print("畳み込みの層数  普通の全畳み込み CNN:", plain.count("conv"), "  U-Net:", unet.count("conv"))
print("受容野（画素）  普通の全畳み込み CNN:", receptive_field(plain), "  U-Net:", receptive_field(unet))
print("→ 同じくらいの層数でも、U-Net は縮小のおかげで、約3倍広い範囲を見て 1 画素を決められる")

**このプロジェクトで U-Net を選んだ理由**: ①降水は画像として見るのが自然、②約 7 千時間分（297日）のデータでも学べる大きさ、③仕組みが比較的読みやすい。
一方、**単純な方法（QM のような表引き）が強雨では U-Net より効いた**ように、複雑な方法が常に良いわけではありません。**基準と比べて、差を区間で確かめる**ことが大切です。

## 4. 乗法的補正と、対数空間の残差学習

### 残差学習とは
予測を**ゼロから**作らず、「ベース場からの**ずれ（残差）**だけを学ぶ」やり方です。
ベース場がすでに粗い予報の総量を持っているので、U-Net は**微調整**に専念できて、学習が楽になります。

### 対数空間で、掛け算で補正する（乗法的補正）
降水は強度の幅が広い（0.1〜50 mm/h）ので、**掛け算（倍率）**で補正するのが自然です。
- U-Net が出すのは倍率の対数 `r`。 **予測 = ベース場 × exp(r)**。 `r = 0` なら倍率 1（そのまま）。
- 出力層は**ゼロ初期化**するので、学習開始時の予測は**ベース場そのもの**になる（悪化しない所から出発する）。
- `r` は `tanh` で ±3 に制限（倍率は約 1/20〜20 まで）。極端な暴走を防ぐ。
- **ベース場が 0 の所は 0 のまま**（0 に何を掛けても 0）。雨のない所に雨を作らない長所だが、後で見る**弱点**でもある。

In [ ]:
from model import MultiplicativeCorrector

m = MultiplicativeCorrector(base=8, depth=3)
b = torch.tensor(regrid_conservative(coarse[idx[0]], (64, 64)), dtype=torch.float32)[None, None]
print("学習前の出力はベース場そのもの:", torch.allclose(m(b), b))

for p_ in m.parameters():                      # 重みを乱して、補正が効いた状態にする
    nn.init.normal_(p_, std=0.1)
out = m(b)
wet = b > 0.05
ratio = (out[wet] / b[wet])
print("補正後の倍率の範囲: %.2f 〜 %.2f（上限は e^3 ≒ 20）" % (ratio.min(), ratio.max()))
print("ベース場が0の画素の出力は全て0か:", bool((out[b == 0] == 0).all()))

## 5. 損失関数と「総量が落ちる」問題

**損失関数**は「予測がどれだけ外れているか」を 1 つの数にしたものです。学習は、この数が小さくなるように重みを少しずつ動かす（**勾配降下法**）作業です。何を損失にするかで、モデルの性格が決まります。

| 損失 | 内容 | 傾向 |
|---|---|---|
| **対数 MSE** | `log(予測+ε)` と `log(真値+ε)` の二乗誤差 | 強い雨が損失を支配するのを防ぐ。ただし**中央値を狙う**ので総量が落ちる |
| **Tweedie 偏差** | ゼロを含む非負データ向けの尤度に基づく損失 | **平均を狙う**ので総量が合う（降水の標準的な選択肢） |

> **なぜ対数だと総量が落ちるのか**: 分布が歪んでいると、中央値は平均より小さい。対数空間の誤差を最小にする予測は（元の単位では）中央値寄りになり、まれな強雨を「無視」するので、**面積平均が過小**になります。

実際に、同じ合成データで 2 つのモデルを学習して比べます（数十秒〜数分）。
- A: 乗法のみ ＋ 対数 MSE（最初の素朴な構成）
- B: 乗法 ＋ **加法の項** ＋ Tweedie（最終的な既定の構成）。加法の項 `softplus(a)` は、ベース場が 0 の所にも雨を出せるようにする足し算で、ベース場の位置ずれ・取りこぼしを救う。

In [ ]:
import time
from model import HybridCorrector, log_loss, tweedie_loss
from train import log_rmse

FACTOR, N = 8, 64
c_tr, f_tr = make_dataset(300, N, FACTOR, seed=0)
c_te, f_te = make_dataset(100, N, FACTOR, seed=10_000)       # 学習とは別の乱数列（テスト）
base_tr, base_te = regrid_conservative(c_tr, (N, N)), regrid_conservative(c_te, (N, N))
T = lambda a: torch.tensor(a[:, None], dtype=torch.float32)
xb, yb = T(base_tr), T(f_tr)

def fit(model, step_loss, epochs=12, lr=2e-3, bs=16, seed=0):
    """ミニバッチ勾配降下（AdamW）で学習する。"""
    torch.manual_seed(seed)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    for ep in range(epochs):
        perm = torch.randperm(len(xb))
        for i in range(0, len(xb), bs):
            ids = perm[i:i + bs]
            loss = step_loss(model, xb[ids], yb[ids])
            opt.zero_grad(); loss.backward(); opt.step()
    return model.eval()

def predict(model, x):
    with torch.no_grad():
        out = model(T(x))
    return (out[0] if isinstance(out, tuple) else out).numpy()[:, 0]

t0 = time.time()
model_a = fit(MultiplicativeCorrector(base=12, depth=3), lambda m, x, y: log_loss(m(x), y))
model_b = fit(HybridCorrector(base=12, depth=3), lambda m, x, y: tweedie_loss(m(x)[0], y))
print(f"学習時間 {time.time() - t0:.0f} 秒")

preds = {"基準（ベース場のみ）": base_te, "A: 乗法 + 対数MSE": predict(model_a, base_te), "B: ハイブリッド + Tweedie": predict(model_b, base_te)}
print(f"{'手法':<24}{'対数RMSE':>10}{'総量比(予測/真値)':>20}")
for k, p_ in preds.items():
    print(f"{k:<22}{log_rmse(p_, f_te).mean():>10.3f}{p_.sum() / f_te.sum():>20.2f}")

**読み方**
- **対数 RMSE**（小さいほど良い）: 対数 MSE で学習した A は、対数の物差しで測るので良い値が出やすい。
- **総量比**（1.0 が理想）: A は 1 を大きく下回る（雨を減らして対数誤差を下げている）。B は 1 に近い。

つまり**対数 RMSE だけで「改善した」と言ってはいけない**、というのが本プロジェクトの大事な教訓です。必ず総量と併記します。

In [ ]:
i = idx[1]
fig, ax = plt.subplots(1, 4, figsize=(15, 4))
for a_, d, t in zip(ax, [f_te[i], base_te[i], preds["A: 乗法 + 対数MSE"][i], preds["B: ハイブリッド + Tweedie"][i]],
                    ["真の降水", "ベース場", "A: 乗法+対数MSE", "B: ハイブリッド+Tweedie"]):
    a_.imshow(d, vmin=0, vmax=15, cmap="Blues"); a_.set_title(t); a_.set_xticks([]); a_.set_yticks([])
plt.show()

## 6. 評価指標

1 つの指標では見落とすものがあるので、目的の違う指標を組み合わせます。

| 指標 | 何を見るか | 理想 |
|---|---|---|
| **対数 RMSE** | 弱い雨も含めた画素ごとの誤差（対数空間） | 小さいほど良い |
| **総量比** | 領域全体の降水の合計 予測/真値 | 1 |
| **POD**（捕捉率） | 実際に雨があった所のうち、予測も雨だった割合 | 1 |
| **FAR**（空振り率） | 予測が雨だった所のうち、実際は降らなかった割合 | 0 |
| **CSI**（臨界成功指数） | 的中 ÷（的中＋空振り＋見逃し）。しきい値以上を「事象」とする | 1 |
| **頻度バイアス** | 予測の事象数 ÷ 真値の事象数 | 1 |
| **FSS**（Fractions Skill Score） | 窓内の「しきい値以上の画素の割合」を比べる。**位置が少しずれても近ければ点がもらえる** | 1 |

しきい値（例: 1・5・10 mm/h）ごとに見ると、**強雨の技能**が分かります。

In [ ]:
from metrics_ext import categorical, fss_window

print(f"{'手法':<24}" + "".join(f"{'CSI@' + str(t):>9}{'POD@' + str(t):>9}" for t in (1, 5)) + f"{'頻度比@5':>10}")
for k, p_ in preds.items():
    row = f"{k:<22}"
    for t in (1, 5):
        c = categorical(p_, f_te, t)
        row += f"{c['csi']:>9.3f}{c['pod']:>9.3f}"
    row += f"{categorical(p_, f_te, 5)['fbias']:>10.2f}"
    print(row)

### FSS: 「少しのずれ」を許す理由
降水の細かい位置を完全に当てるのは難しいので、**窓（近傍）の広さ**を変えて見ます。窓 1 は画素どうしの厳しい比較、窓が大きいほど位置のずれに寛容です。
下の例では、雨の塊を右にずらした予測を、窓の広さを変えて評価しています。

In [ ]:
t_ = np.zeros((1, 64, 64)); t_[0, 28:36, 28:36] = 8.0         # 真の雨の塊
for shift in (0, 3, 6, 12):
    p_ = np.roll(t_, shift, axis=2)
    print(f"ずれ {shift:>2} 画素:  " + "  ".join(f"窓{w:>2}: FSS={fss_window(p_, t_, 5.0, w):.2f}" for w in (1, 9, 17, 33)))
print("→ ずれが大きいほど FSS は下がり、窓を広げるほど寛容になる")

### 6-1. 指標のくわしい解説

#### 分割表（的中・空振り・見逃し）
しきい値（例: 5 mm/h）以上を「事象あり」として、予測と真値の組み合わせを 4 つに分けます。

| | 真値: 事象あり | 真値: 事象なし |
|---|---|---|
| **予測: あり** | **的中**（H） | **空振り**（F） |
| **予測: なし** | **見逃し**（M） | 的中でない「なし」（使わない） |

- **POD（捕捉率）** = H ÷ (H + M)。実際にあった事象のうち、当てた割合。1 が理想。
- **FAR（空振り率）** = F ÷ (H + F)。「あり」と予測したうち、外れた割合。0 が理想。
- **CSI（臨界成功指数）** = H ÷ (H + F + M)。当たり・空振り・見逃しをまとめた点数。1 が理想。「なし」の的中は数えないので、稀な現象でも甘くならない。
- **BI（頻度バイアス）** = (H + F) ÷ (H + M)。予測の事象数 ÷ 真値の事象数。1 が理想。1 より大きい＝出しすぎ、1 より小さい＝出なさすぎ。

#### RMSE と対数 RMSE
- **RMSE** = √(誤差の二乗の平均)。大きな誤差を強く罰する。降水では、少数の強雨の誤差だけで決まり、弱い雨の良し悪しが見えにくい。
- **対数 RMSE**: log(降水 + 0.1) の空間で RMSE を取る。弱い雨も含めた「割合の誤差」を見る。0.1 は 0 の対数を避ける小さな値。総量の偏りは見えにくいので、**総量比と併せて見る**。

#### FSS（Fractions Skill Score）
窓（近傍）ごとに「しきい値以上の画素の割合」を求め、予測と真値で比べます。
FSS = 1 − 〔割合の差の二乗の平均〕 ÷ 〔予測の割合の二乗の平均 + 真値の割合の二乗の平均〕。位置が少しずれても、近くにあれば点がもらえます（窓が大きいほど寛容）。

In [ ]:
# 小さな例で、分割表から POD・FAR・CSI・BI を手計算して、関数と比べる
rng = np.random.default_rng(0)
truth = (rng.random((50, 50)) < 0.08)                 # 真値: 8% の画素に事象
pred = truth.copy()
flip_off = truth & (rng.random(truth.shape) < 0.4)    # 見逃し: 事象の 40% を取りこぼす
flip_on = (~truth) & (rng.random(truth.shape) < 0.03) # 空振り: 事象のない所の 3% に出す
pred = (pred & ~flip_off) | flip_on
H = int((pred & truth).sum()); F_ = int((pred & ~truth).sum()); M = int((~pred & truth).sum())
print(f"的中 H={H}  空振り F={F_}  見逃し M={M}")
print(f"POD={H / (H + M):.3f}  FAR={F_ / (H + F_):.3f}  CSI={H / (H + F_ + M):.3f}  BI={(H + F_) / (H + M):.3f}")
c = categorical(pred.astype(float), truth.astype(float), 0.5)
print(f"関数 categorical: POD={c['pod']:.3f}  CSI={c['csi']:.3f}  BI={c['fbias']:.3f}  （手計算と一致）")

#### CSI は「出しすぎ」に有利（頻度バイアスと一緒に見る理由）
予測の強さに少しノイズがある場合、「事象あり」とするしきい値を**低くする**（＝事象を多めに出す）と、的中が増えて CSI が上がります（空振りも増えます）。
そのため、**CSI だけで方法を比べると、多めに出す方法が有利に見えます**（出しすぎると空振りが増えて、いずれ下がります）。必ず BI（頻度バイアス）も併記し、BI がそろった状態で比べます。

In [ ]:
rng = np.random.default_rng(1)
truth = rng.random(200000) < 0.02                      # 稀な事象（2%）
score = truth * 1.5 + rng.normal(size=truth.size)       # 事象があるとスコアが高くなるが、ノイズがある予測
print(f"{'しきい値':>8}{'BI':>8}{'POD':>8}{'FAR':>8}{'CSI':>8}")
for thr in (3.0, 2.5, 2.0, 1.5, 1.0, 0.5):
    pr = score >= thr
    H, F_, M = (pr & truth).sum(), (pr & ~truth).sum(), (~pr & truth).sum()
    print(f"{thr:>8.1f}{(H + F_) / (H + M):>8.2f}{H / (H + M):>8.2f}{F_ / (H + F_):>8.2f}{H / (H + F_ + M):>8.3f}")
print("→ BI を 1 より大きくすると CSI は上がる（ここでは BI≈1.4 の行が最大）。その分 FAR が悪化し、出しすぎるとまた下がる。BI がそろった行どうしで比べるのが公平")

## 7. ブートストラップ（信頼区間）

テスト 100 件で「A より B の方が 0.03 良い」と出ても、**それが偶然か**は分かりません。別の 100 件なら逆かもしれません。
**ブートストラップ**は、手元のデータから「**重複を許して**同じ数だけ取り直す」を何千回も繰り返し、指標のばらつきを見積もる方法です。
そこから **95% 信頼区間**（100 回中 95 回は入る範囲）が得られます。**差の区間が 0 を含まなければ、差は誤差でないと考えます。**

In [ ]:
from train import bootstrap_ci, paired_diff_ci

r_base = log_rmse(preds["基準（ベース場のみ）"], f_te)
r_b = log_rmse(preds["B: ハイブリッド + Tweedie"], f_te)
m, lo, hi = bootstrap_ci(r_b, n_boot=2000)
print(f"B の対数RMSE: {m:.3f}  95%区間 [{lo:.3f}, {hi:.3f}]")
d, lo, hi = paired_diff_ci(r_b, r_base, n_boot=2000)       # 同じ事例どうしの差（対の差）
print(f"B − 基準 の差（負なら改善）: {d:+.3f}  95%区間 [{lo:+.3f}, {hi:+.3f}]  → " + ("差は誤差でない" if (hi < 0 or lo > 0) else "差は誤差の範囲"))

### 注意①: 「対の差」で比べる
手法 A と B を**同じ事例**で比べるなら、事例ごとの差を取ってからブートストラップします（事例の難しさが相殺されて、差が見えやすくなる）。

### 注意②: 事例が独立でないとき — ブロック・ブートストラップ
時系列の降水は、**今日と明日が似ている**（時間相関）。1 日ずつバラバラに取り直すと「独立な事例が多い」と思い込み、**区間が狭すぎ**になります。
そこで**連続した数日をひとまとまり（ブロック）**にして取り直します。実データではこれが必須です。

In [ ]:
rng = np.random.default_rng(0)
# 時間相関のある系列（AR(1): 前の値の 0.9 倍＋ノイズ）を作る。400 件あっても、実質の独立な件数はずっと少ない
x = np.zeros(400)
for i in range(1, 400):
    x[i] = 0.9 * x[i - 1] + rng.normal()
for block in (1, 10, 30):
    m, lo, hi = bootstrap_ci(x, n_boot=2000, block=block)
    print(f"ブロック長 {block:>2}: 平均 {m:+.2f}  95%区間 [{lo:+.2f}, {hi:+.2f}]  幅 {hi - lo:.2f}")
print("→ ブロックを長くするほど区間が広がる（相関を考慮した、より正直な不確かさ）")

### 注意③: 学習のばらつきも含める（二段ブートストラップ）
ニューラルネットは、**乱数シード（初期値・バッチの順）**が違うと結果が少し変わります。
事例だけのブートストラップは**この学習のばらつきを含まない**ので、区間が狭すぎて「偽の差」を拾います（Phase 3 で実際に起きました）。
→ 「学習シード」と「事例」の**二段**で取り直す（`src/boot.py`）。有望に見えた差は、**選ばなかった別のシードで再確認**する。

In [ ]:
# 学習シードのぶれを体感する: 同じ構成で、シードだけ変えて3回学習
rm = []
for s in (1, 2, 3):
    mdl = fit(HybridCorrector(base=12, depth=3), lambda m, x, y: tweedie_loss(m(x)[0], y), epochs=8, seed=s)
    p_ = predict(mdl, base_te)
    rm.append((log_rmse(p_, f_te).mean(), categorical(p_, f_te, 5)['csi']))
print("シード  対数RMSE   CSI@5")
for s, (a_, b_) in zip((1, 2, 3), rm):
    print(f"{s:>5}{a_:>10.3f}{b_:>8.3f}")
print("→ 設定を変えていないのに数字が動く。この幅より小さい差は『差がある』と言えない")

## 8. 実データ（GSM × GSMaP）での注意点

合成データで手法の挙動を学んだあと、実データへ進みます（現在 Phase 5）。

| | 内容 |
|---|---|
| **入力（予報）** | 気象庁 **GSM**（全球数値予報モデル）日本域。格子は緯度 0.1°×経度 0.125°。降水は「初期時刻からの積算」なので、**隣り合う時刻の差**で 1 時間降水にする |
| **真値** | **GSMaP**（JAXA の衛星＋雨量計の降水推定）1 時間降水。格子 0.1°。真値と言っても**別の推定値**で、誤差・欠測がある |
| **領域・期間** | 沖縄周辺（122〜132E・22〜30N）、2025-04-01〜2026-05-10 の約 400 日 |

### 合成データとの違い
- 解像度が**同程度**なので、8 倍の細分化ではなく、**バイアスと位置の補正**が中心になる。
- **欠測**がある（GSMaP の NaN・全面 −99 の日）。損失・評価では欠測画素を除く。
- **強雨が少ない**: 10 mm/h 以上の画素は 0.2%。独立な強雨の事例は数十エピソード → 評価の区間は広くなる。
- **時間相関**がある → ランダム分割は禁止。時間のまとまり（ブロック）で分ける。

### ブロック巡回による分割
**データリーク**（学習に使った情報が評価に漏れること）を避けるため、10 日のブロックごとに役割を割り当て、ブロックの間に 1 日の空けを置きます。
10 ブロックの繰り返しで 学習 7・検証 1・テスト 2。**テスト集合は最終評価の 1 回だけ**使い、結果を見て設定を変えません。

In [ ]:
from datetime import date, timedelta
from split import ORIGIN, role

colors = {"train": "#4c78a8", "val": "#f58518", "test": "#e45756", "gap": "#dddddd"}
days = [ORIGIN + timedelta(days=i) for i in range(402)]
fig, ax = plt.subplots(figsize=(13, 1.8))
for i, d in enumerate(days):
    ax.barh(0, 1, left=i, color=colors[role(d)], edgecolor="none")
ax.set_yticks([]); ax.set_xlabel("2025-04-01 からの日数")
for k, c in colors.items():
    ax.bar(0, 0, color=c, label={"train": "学習", "val": "検証", "test": "テスト", "gap": "空け"}[k])
ax.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, 1.6)); plt.show()

### 実データでの基準（GSM そのもの）
補正モデルを作る前に、まず**基準**として「GSM をそのまま使ったとき」の指標を測ります（`src/real_baseline.py`・テスト集合は使わない）。
補正モデルの価値は、**この基準との差が、ブロック単位のブートストラップ区間で 0 を含まないか**で判断します。
結果は `precip-unet-lab.md` の記録を参照してください。

### 8-1. 後処理 QM（分位点写像）と、期間のずれ
**QM（分位点写像）**は、「予報値を小さい順に並べたときの p% 点は、観測では何 mm か」を表にして、予報値を機械的に置き換える後処理です（機械学習ではなく表引き）。
予報の**強さの分布を観測の分布に合わせる**ので、強い雨の頻度が戻ります。ただし、**写像を作った期間の分布に合わせた表**なので、**期間の性質が変わる（雨の多い／少ない）と総量が外れます**。
予報と観測の関係が期間で変わる（多雨期は弱めに出す／少雨期は偏りがない）と、古い表は出しすぎます。対策は、**直近の期間で写像を定期的に作り直す**ことです。

In [ ]:
rng = np.random.default_rng(0)
def make_period(scale, bias, n=50000):                    # 観測と予報（予報は観測を bias 倍・ノイズ付きで出す）
    obs = rng.exponential(scale, n) * (rng.random(n) < 0.3)
    fc = obs * bias * rng.lognormal(0, 0.3, n)
    return fc, obs
levels = np.linspace(0, 0.999, 1000)
def fit(fc, obs):
    xp = np.quantile(fc, levels); fp = np.quantile(obs, levels)
    xp, i = np.unique(xp, return_index=True); return xp, fp[i]
apply = lambda m, a: np.interp(a, m[0], m[1])

fc_a, obs_a = make_period(2.0, 0.6)     # 雨の多い期間: 予報は雨を弱めに出す（0.6倍）→ 写像は「強める」表になる
fc_b, obs_b = make_period(0.7, 1.0)     # 雨の少ない期間: 予報は偏っていない → 古い表で強めると出しすぎる
m_old = fit(fc_a, obs_a)
m_new = fit(fc_b[:25000], obs_b[:25000])      # 直近の期間の前半で作り直す
print("総量比（予測の合計 ÷ 観測の合計。1が理想）")
print(f"  QM なし            : {fc_b[25000:].sum() / obs_b[25000:].sum():.2f}")
print(f"  古い写像（多雨期で作成）: {apply(m_old, fc_b[25000:]).sum() / obs_b[25000:].sum():.2f}")
print(f"  更新した写像（直近で作成）: {apply(m_new, fc_b[25000:]).sum() / obs_b[25000:].sum():.2f}")

## 9. 用語集

| 用語 | 意味 |
|---|---|
| **mm/h** | 1 時間あたりの降水量。1 mm/h 以上＝「雨」、10 mm/h 以上＝強い雨、の目安 |
| **格子（グリッド）** | 領域を碁盤の目に区切った、1 マスごとの値の並び |
| **ダウンスケーリング** | 粗い格子の情報から、細かい格子の分布を推定すること |
| **バイアス補正** | 予報が系統的に雨を多く／少なく出す癖を直すこと |
| **再格子化** | 格子の細かさ・位置を貼り替えること |
| **保存的再格子化** | 重なり面積で重み付け平均し、総量を保つ再格子化 |
| **双線形補間** | 周囲 4 点から直線的に補う補間。降水の総量は保たない |
| **ベース場** | 粗い予報を保存的に細かい格子に載せたもの。補正の出発点 |
| **U-Net** | 縮めて戻す U 字のネットワーク。スキップ結合で細部を保つ |
| **畳み込み（3×3）** | 小さな窓をずらして重みとの積和を取る操作。局所の関係を捉える |
| **GroupNorm** | チャンネルをグループに分けて平均 0・分散 1 にそろえる。小バッチでも安定 |
| **SiLU** | `x·sigmoid(x)`。なめらかな活性化関数 |
| **活性化関数** | 層の間に挟む非線形関数。これがないと層を重ねる意味がない |
| **平均プーリング** | 範囲を平均して縮める。視野を広げる |
| **スキップ結合** | 縮める前の特徴を、戻した特徴に連結して細部を渡す |
| **残差学習** | 出力そのものでなく、ベース場からの「ずれ」だけを学ぶ |
| **乗法的補正** | 予測 = ベース場 × exp(r)。倍率で補正する。0 は 0 のまま |
| **加法の項** | `softplus(a)` を足す項。ベース場が 0 の所にも雨を出せる |
| **損失関数** | 予測の外れ具合を 1 つの数にしたもの。学習はこれを小さくする |
| **対数 MSE** | 対数空間の二乗誤差。中央値寄りで総量が落ちる |
| **Tweedie 偏差** | ゼロを含む非負データ向けの損失。平均を狙う |
| **勾配降下法 / AdamW** | 損失が小さくなる向きへ重みを少しずつ動かす最適化法 |
| **エポック / バッチ** | 全データを 1 周するのが 1 エポック。重み更新は小さなまとまり（バッチ）ごと |
| **過学習** | 学習データには合うが、新しいデータでは外れること |
| **総量比** | 領域の降水合計の 予測/真値。1 が理想 |
| **POD / FAR / CSI** | 捕捉率 / 空振り率 / 臨界成功指数。しきい値以上を事象として評価 |
| **頻度バイアス** | 予測の事象数 ÷ 真値の事象数。1 が理想 |
| **FSS** | 窓内の事象割合を比べる指標。位置の小さなずれに寛容 |
| **ブートストラップ** | 重複を許して取り直し、指標のばらつきを見積もる方法 |
| **信頼区間（95%）** | 指標が入ると見積もられる範囲。差の区間が 0 を含まなければ差は誤差でない |
| **ブロック・ブートストラップ** | 連続数日などのまとまりで取り直す。時間相関がある場合に必須 |
| **二段ブートストラップ** | 学習シードと事例の両方を取り直し、学習のばらつきも含める |
| **データリーク** | 学習に使った情報が評価に漏れること。時間順・ブロック分割で防ぐ |
| **CNN** | 畳み込み層を重ねたネットワークの総称。U-Net は CNN の一種 |
| **受容野** | 出力の 1 画素が、入力のどの範囲を見て決まっているか。縮小（プーリング）で広がる |
| **分割表** | 予測と真値の「あり／なし」の 4 通り（的中・空振り・見逃し・なしの的中）を数えた表 |
| **BI（頻度バイアス）** | (的中 + 空振り) ÷ (的中 + 見逃し)。1 より大きい＝出しすぎ。CSI を読むときは併記する |
| **RMSE** | 誤差の二乗平均の平方根。大きな誤差を強く罰する |
| **欠測マスク** | 値のない画素の印。損失・評価・QM から除く（0 mm/h とみなさない） |
| **欠測を考慮した補間** | 有効な点だけで重みを再正規化する補間。合計は保存しない |
| **QM（分位点写像）** | 予報の分布を観測の分布に合わせる表引きの後処理。期間の性質が変わると総量が外れる |
| **GSM / GSMaP** | 気象庁の全球数値予報モデル / JAXA の衛星降水推定 |
| **ベースライン（基準）** | 比較の物差しとなる単純な方法。補正の価値は基準との差で測る |

## まとめ
1. 降水は**ゼロが多く歪んでいる**ので、再格子化は**保存的**に、補正は**倍率（乗法）**で行う。
2. 損失は物差し。**対数だけだと総量が落ちる**ので、平均を狙う損失や加法の項を使い、**複数の指標で**見る。
3. 「良くなった」は、**基準との差と、ブートストラップ区間**（時間相関はブロックで、学習のばらつきは複数シードで）で判断する。
4. 実データでは、欠測・時間相関・少ない強雨に注意し、**テストは最後に 1 回だけ**。
5. **CSI は出しすぎに有利**。**BI（頻度バイアス）も併記**して比べる。QM は期間のずれに弱いので、**直近の期間で更新**する。